# HABS Neuro-AI Hackathon 2026: Team 12 results walkthrough

This notebook reproduces the historical Study 01–10 tables and figures from `results/`. For Study 11, see [the revised article](../paper/submission_paper.pdf), [its supplement](../paper/supplement.pdf), and [the numerical provenance](../results/study11/provenance.json). All numbers come from nested ID-by-theme cross-validation, fold-isolated scaling, and post-release forward-only filtering across 314 participant IDs and 22,974 trials.

**Research question:** How do cortical activity (EEG) and peripheral autonomic responses (ECG, EDA, respiration) unfold during acute distress, how do leakage controls and cross-day drift constrain multimodal decoding, and does early EEG add prospective information about future autonomic reactivity beyond past body dynamics?


In [1]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

from habs_stress.figures import generate_all

ROOT = Path('..').resolve() if Path('../results').is_dir() else Path('.').resolve()
RESULTS = ROOT / 'results'
FIGS = ROOT / 'paper' / 'figures'

ph = json.loads((RESULTS / 'physiology.json').read_text())
cf = json.loads((RESULTS / 'causal_forecasting.json').read_text())
gc = json.loads((RESULTS / 'generalization_clinical.json').read_text())
print('Loaded verified aggregates from:', RESULTS)


## 1. Acute stress physiology and ECG morphology

Lead I ECG polarity was aligned by recording (96.6% of trial rows flagged for inversion). Beat-by-beat P-QRS-T delineation shows RR lengthening (d = +0.116), no detected PR-segment shift (d = -0.005), and a Bazett QTc drop (standardized beta = -0.088) that disappears after delta-RR adjustment (beta = -0.007), compatible with vagally mediated cardiac deceleration. Raw CURATED (+0.038 [-0.001, +0.077]) and ICA-preprocessed (+0.039 [-0.002, +0.078]) heartbeat-evoked potentials both span zero.


In [2]:
paths = generate_all(RESULTS, FIGS)
display(Image(filename=str(FIGS / 'fig1_physiology_cascade.png'), width=900))


## 2. Exploratory prospective early EEG to future physiology (t = 5.000 s post-release boundary)

No post-release sample at or after t = 5.000 s touches any predictor (upstream HABS PREPROCESSED zero-phase FIR and full-recording Picard ICA remain an open check from raw EEG). Trial-matched early EEG (REVE-base) adds a small increment to future heart rate change prediction over past body dynamics (M0 r = 0.685, RMSE 5.540 bpm; M2_resid +0.009, -0.077 bpm RMSE; M2_joint +0.011, -0.086 bpm RMSE, 71.4% with dr > 0; internal 5-cohort LOCO without theme mask +0.011, 5/5 positive point estimates, 3/5 CIs > 0). It persists with nine artefact covariates, 46 interior channels, clean trials (n = 207), and outperforms permuting EEG across same-video trials (NC2: -0.015; own minus permuted +0.024). Adding EEG degrades skin conductance forecasting (-0.010 resid, -0.030 joint) and shows no detectable change on respiration (+0.001).


In [3]:
pri = cf['audit_forecast']['module2_primary_forecasting']['primary_cold_start_3x3']
rows = []
for name, key in [
    ('Heart rate [5, 10) s', 'tgt_ecg_hr_chg_5_10'),
    ('Skin conductance [5, 10) s', 'tgt_eda_sc_chg_5_10'),
    ('Respiration [5, 10) s', 'tgt_resp_amp_chg_5_10'),
]:
    rows.append({
        'Target': name,
        'M0 body-only (r)': round(pri[key]['models']['M0_linear']['r']['mean'], 4),
        'M2 resid minus M0': round(pri[key]['paired_vs_m0_linear']['M2_reve_resid']['delta_r']['mean'], 4),
        'M2 joint minus M0': round(pri[key]['paired_vs_m0_linear']['M2_reve_joint']['delta_r']['mean'], 4),
        'GBDT M2 minus M0': round(pri[key]['paired_m2_gbdt_vs_m0_gbdt']['delta_r']['mean'], 4),
    })
display(pd.DataFrame(rows))
display(Image(filename=str(FIGS / 'fig2_causal_brain_heart.png'), width=900))


## 3. Generalization ladder, cross-day drift, and k-shot personalization

Moving from random splits (R1: 0.589) to unseen IDs and unseen themes (R4: 0.569) removes 0.020 AUC; the step masking video themes represents 70-79% of the observed drop under varying training sizes. Across a one-week gap, EDA-only decoding drops by -0.039 paired AUC (N = 103; unpaired 0.560 -> 0.523) while no drop is detected for EEG (+0.025 [-0.006, +0.055]). Naive personal Ridge VA retraining lags Cold-Start through k = 40 (-0.053), whereas EB-MAP stays flat (-0.001).


In [4]:
display(Image(filename=str(FIGS / 'fig3_generalization_drift.png'), width=950))


## 4. Varimax PCA, coherence, and offline known-class averaging

Varimax-rotated PCA on the correlation matrix across 314 IDs (16,276 complete trials; 53.3% variance explained) separates skin conductance (F1), EEG-cardiorespiratory covariance (F2), and subjective-cardiorespiratory ratings (F3). Neuro-autonomic coherence is modest (median rho = +0.111, 21.0% <= 0; 1-week stability r = 0.066 [-0.087, +0.211], p = 0.39). Offline averaging of K = 1..10 known-class Cold-Start R4 clips raises AUC to 0.680-0.698 (disjoint same-class blocks checked for K <= 5), with per-ID oracle sensitivity at 90% specificity reaching 41.8%, 45.5%, and 52.8% (an offline label-grouped exercise rather than real-time transition detection).


In [5]:
m10e = gc['module10e_clinical_monitoring']
mon_rows = []
for lbl, key in [
    ('Standard stress vs. other', 'stress_standard_vs_nonstress'),
    ('Acute threat vs. calm/neutral', 'acute_threat_vs_calm_neutral'),
    ('Strict stress (3/7) vs. relaxed', 'stress_strict37_vs_relaxed64'),
]:
    bk = m10e[key]['S07_enriched']['by_k']
    mon_rows.append({
        'Clinical contrast': lbl,
        '10 s AUC': round(bk['1']['auc']['mean'], 4),
        '100 s AUC': round(bk['10']['auc']['mean'], 4),
        '10 s Sens@90%Spec': f"{100 * bk['1']['sens_at_spec90']['mean']:.1f}%",
        '100 s Sens@90%Spec': f"{100 * bk['10']['sens_at_spec90']['mean']:.1f}%",
    })
display(pd.DataFrame(mon_rows))
display(Image(filename=str(FIGS / 'fig4_coherence_monitoring.png'), width=900))
